# Day 3, afternoon: the live demo (not graded)

A **workbench** next to the demo: you write no code here. The demo itself runs in a terminal:
```
python -m asr_demo.server          # then open http://localhost:8000 (Chrome, Firefox, Safari, Edge)
```
The figures follow lecture L6. The afternoon is not graded; your `d3` decision is (did the demo confirm or
contradict your morning prediction?).

| You implement (stretch) | In | L6 slides | Sections here |
|---|---|---|---|
| `LocalAgreementSession` | `speech/streaming_live.py` | "Your live streamer" | 4 |

2 holes (the commit of an update, the final text at release); it reuses your `local_agreement` of this morning.
Check: `pytest -m stretch`. Your own systems go in `my_backends.py` (L6, "Hijack the demo"): no hole there.

**Plan for the lab (2 h).**

| Time | What | Computer time |
|---|---|---|
| 0:00–0:15 | Start the server; record 3 sentences on the reference (`New prompt` gives a reference, so the page shows a WER) | |
| 0:15–0:30 | Sections 1–2: the contract, what the demo lists and why some entries are unavailable | 1 min |
| 0:30–0:50 | Hijack: register your day-2 decoder in `my_backends.py`, restart the server, section 3 | 1 min |
| 0:50–1:10 | Stretch: `LocalAgreementSession` (2 holes), `pytest -m stretch`, section 4 | 1 min |
| 1:10–1:40 | Record the same sentences on 3–4 backends (Replay button); section 5 replays them all | 2–3 min |
| 1:40–2:00 | Section 6: prediction vs measurement; your d3 `DECISIONS.md` entry; tag `d3` | |

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "speech").is_dir() else Path.cwd().parent  # the folder that contains speech/
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # results/ and data/ are relative to the bundle root
DATA = Path(os.environ.get("DSP_DATA", ROOT / "data"))
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import pandas as pd
from IPython.display import display

from asr_demo import backends
from asr_demo.metrics import run_session_offline
from asr_demo.replay import export_librispeech, replay
from common.bench import load_records
from speech.data import load_librispeech
from speech.streaming_sim import reference_word_ends

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
pd.set_option("display.max_colwidth", 70)
REFERENCE = "reference/fastconformer-114M-int8"


def not_yet(e):
    """What a cell prints when it reaches one of your stubs."""
    print(f"Not implemented yet: {e}\nImplement it in the package, then re-run this cell.")

## 1. The contract, and the four numbers

Every backend answers two calls (L6, "The contract: two methods"): `accept_chunk(audio)` every 100 ms while the
button is held, `finish()` at release. The server, the replay harness and this notebook time them the same way
(`asr_demo/metrics.py`). One LibriSpeech utterance through an offline model and through the reference
transducer:

In [ ]:
utt = load_librispeech(DATA / "LibriSpeech", n=1)[0]
print("reference:", utt["text"].lower(), f"({len(utt['audio']) / utt['sr']:.1f} s)")
rows = []
for name in ("hf/whisper-tiny.en", REFERENCE):
    try:
        m = run_session_offline(backends.get(name), utt["audio"], utt["sr"], utt["text"])
    except backends.BackendUnavailable as e:
        print(e)
        continue
    rows.append(m)
if rows:
    display(pd.DataFrame(rows)[["backend", "streaming", "audio_s", "n_partials", "first_partial_ms", "final_latency_ms",
                                "rtf", "wer", "text"]].round(3))

`first_partial_ms` is counted in audio time here (the replay is faster than speech); `--realtime` in
`python -m asr_demo.replay` feeds the audio at speaking speed, like a microphone.

## 2. What the demo lists

The page shows every registered backend: the provided ones (`asr_demo/backends.py`) and yours
(`my_backends.py`). The server loads a backend the first time it is selected, then keeps it; if it cannot (a
model not downloaded, a package missing, a TODO not implemented yet), the page says why. This cell loads the
CPU-ready speech entries (`whisper-tiny.en`, `wav2vec2-base-960h`, `wav2vec2-base-100h`, transducers, and yours),
so the table says which ones are `ready` and why any others cannot be used. `hf/whisper-base.en+partials`
(~290 MB download, not fetched on day 2, slow on CPU) and day-1 `digits/*` are left unloaded here (remove from
`SKIP_AUTO` to test them).

In [ ]:
SKIP_AUTO = {"hf/whisper-base.en+partials"}  # opt-in: ~290 MB download (not fetched on day 2) and slow on CPU
for name in [b["name"] for b in backends.available()]:
    if name.startswith("digits/") or name in SKIP_AUTO:
        continue
    try:
        backends.get(name)
    except backends.BackendUnavailable:
        pass  # the reason is in the table
listing = pd.DataFrame(backends.available())
listing["status"] = ["ready" if b.loaded else (f"unavailable: {b.error}" if b.error else "not loaded (on demand in demo / opt-in)")
                     for b in listing.itertuples()]
with pd.option_context("display.max_colwidth", None):  # the whole reason, not its first 70 characters
    display(listing[["name", "status", "description"]])

## 3. Your systems in the demo

Uncomment the examples of `my_backends.py` (your day-2 decoder with the $\lambda$, $\beta$, $N_{\mathrm{beam}}$
of your d2 decision; your streaming policy around it) and run this cell: it reloads `my_backends.py` and runs
your `student/...` entries on 3 LibriSpeech utterances (for the live server, restart `python -m asr_demo.server`).

In [ ]:
import importlib
import my_backends

importlib.reload(my_backends)
mine = [b["name"] for b in backends.available() if b["name"].startswith("student/")
        and b["name"] not in ("student/local-agreement(whisper-tiny.en)", "student/local-agreement(wav2vec2-base-960h)")]
wavs = export_librispeech(DATA / "LibriSpeech", ROOT / "results/librispeech_wavs", n=8)
if not mine:
    print("No backend of yours yet: uncomment an example in my_backends.py, then re-run this cell.")
else:
    own = replay(wavs, mine, jsonl=None, limit=3)
    if len(own):
        display(own[["backend", "recording", "audio_s", "final_latency_ms", "rtf", "wer", "text"]].round(3))

## 4. Your live streamer, partial by partial (stretch)

`LocalAgreementBackend` re-decodes the buffer every `step_s` seconds of new audio and shows the committed words
followed by the tentative ones (L6, "Your live streamer"). Below, on the utterance of section 1: what the page
would show after each update of your streamer around wav2vec2 (0.5 s step), next to the reference transducer.
The figure counts the words shown against the words already spoken (from the forced alignment of the morning).

In [ ]:
if "utt" not in globals():
    utt = load_librispeech(DATA / "LibriSpeech", n=1)[0]
try:
    ref_words, ref_ends = reference_word_ends([utt], cache=ROOT / "results/streaming_word_ends.json")[utt["id"]]
except Exception as e:
    ref_words = ref_ends = None
    print(f"Forced alignment unavailable ({type(e).__name__}: {e}); plotting backend partials only.")
fig, ax = plt.subplots(figsize=(7, 4))
if ref_ends is not None:
    t = np.linspace(0, len(utt["audio"]) / utt["sr"], 200)
    ax.plot(t, [np.sum(ref_ends <= x) for x in t], "k:", label="words spoken (reference)")
for name in ("student/local-agreement(wav2vec2-base-960h)", REFERENCE):
    try:
        m = run_session_offline(backends.get(name), utt["audio"], utt["sr"], utt["text"], keep_partials=True)
    except backends.BackendUnavailable as e:
        if "TODO(" in str(e):
            not_yet(e)
        else:
            print(e)
        continue
    except NotImplementedError as e:
        not_yet(e)
        continue
    times = [p[0] for p in m["partials"]]
    ax.step(times, [len(p[1].split()) for p in m["partials"]], where="post", label=name)
    print(f"{name}: {m['n_partials']} partials, final (WER {m['wer']:.3f}): {m['text']}")
    for tp, text in m["partials"][:: max(1, len(m["partials"]) // 5)]:
        print(f"   {tp:5.1f} s  {text}")
ax.set(xlabel="audio received (s)", ylabel="words shown", title="what the page shows while you speak")
ax.legend(fontsize=7)
fig.tight_layout()

## 5. The same recordings on every backend

Comparing backends on different recordings measures your voice, not the backends (L6, "Compare on the same
recordings"). Every recording of the page is saved in `results/demo_recordings/` (with its reference, if you
gave one); this cell replays up to 5 of them (`MAX_RECS`), or 5 LibriSpeech utterances if you have not recorded
yet, on the speech backends registered below (including your `student/...` entries). `python -m asr_demo.replay`
does the same in a terminal and logs the rows. Replay loads backends lazily: the unavailable ones are skipped with
an explicit message, and the available ones are compared on the exact same recordings.

The `digits/...` backends (Day-1 digit prompts) and `hf/whisper-base.en+partials` (~290 MB download, slow on
CPU) are excluded by default; remove `hf/whisper-base.en+partials` from `SKIP_REPLAY` if you want to include it.

In [ ]:
SKIP_REPLAY = {"hf/whisper-base.en+partials"}
REPLAY_ON = [b["name"] for b in backends.available()
             if not b["name"].startswith("digits/") and b["name"] not in SKIP_REPLAY]
recs = ROOT / "results/demo_recordings"
source = recs if recs.is_dir() and any(recs.glob("*.wav")) else wavs
MAX_RECS = 5  # cap so replay finishes in ~2 min; set to None to replay all recordings
print("replaying", source)
both = replay(source, REPLAY_ON, jsonl=None, limit=MAX_RECS)
if len(both):
    table = both.groupby("backend", sort=False)[["first_partial_ms", "final_latency_ms", "rtf", "wer"]].median()
    display(table.round(3))
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, g in both.groupby("backend", sort=False):
        ax.plot(g.final_latency_ms, 100 * g.wer.astype(float), "o", label=name)
    ax.set(xscale="log", xlabel="wait after release (ms)", ylabel="WER (%)", title=f"replay of {source.name}")
    ax.legend(fontsize=7)
    fig.tight_layout()

**What to listen for** (L6, "What to listen for, and what the numbers miss"): words lost at the start (the
20M transducer, compare `lead_s` in `my_backends.py`), the digit models on your microphone, fluent but wrong
sentences, text that appears only at release.

## 6. Did your prediction hold?

This morning you predicted the wait after release from the wait rate (offline and re-decoding:
$\mathrm{rate}\times\mathrm{dur}$) or as a constant (native streaming). The live `demo` rows of
`results/results.jsonl` (`cfg_mode == "live"`) are compared against the prediction from the replays of section 5.

In [ ]:
def _is_native(name, fallback_streaming):
    b = backends._CACHE.get(name)
    if b is not None:
        return bool(b.streaming and not hasattr(b, "inner"))
    return bool(fallback_streaming)


res = ROOT / "results/results.jsonl"
df = load_records(res) if res.exists() else None
if df is not None and "task" in df:
    live = df[df.task == "demo"].copy()
    if "cfg_mode" in live.columns:
        live = live[live.cfg_mode == "live"].copy()
else:
    live = pd.DataFrame()
if not len(live) or "both" not in globals() or not len(both):
    print("No live request logged yet (task 'demo', cfg_mode 'live'): record with the server running, then re-run this cell.")
else:
    wait_rate = both.assign(rate=both.final_latency_ms / both.audio_s.replace(0, np.nan)).groupby("backend").rate.median()
    stream_wait = both.groupby("backend").final_latency_ms.median()
    streaming = both.groupby("backend").streaming.first()
    live = live[live.model.isin(wait_rate.index)].copy()
    if not len(live):
        print("No live request matches the backends replayed in section 5 (check SKIP_REPLAY / digits/*).")
    else:
        live["predicted_ms"] = [
            stream_wait[m] if _is_native(m, streaming[m]) else wait_rate[m] * a
            for m, a in zip(live.model, live.audio_s)
        ]
        cols = ["model", "audio_s", "first_partial_ms", "final_latency_ms", "predicted_ms", "rtf", "wer"]
        display(live[[c for c in cols if c in live.columns]].round(3))
        fig, ax = plt.subplots(figsize=(5, 5))
        for name, g in live.groupby("model"):
            ax.plot(g.predicted_ms, g.final_latency_ms, "o", label=name)
        top = max(1.0, float(live.final_latency_ms.max()), float(live.predicted_ms.max())) * 1.5
        lim = [1, top]
        ax.plot(lim, lim, "k:", lw=1)
        ax.set(xscale="log", yscale="log", xlabel="predicted wait (ms)", ylabel="measured wait, live (ms)")
        ax.legend(fontsize=7)
        fig.tight_layout()

## Before you leave

Complete your d3 `DECISIONS.md` entry: the system you deploy, your budget, your prediction, what the demo
measured, and why they differ if they do (first use and load time, other programs, the length of your
requests, your microphone). Then tag `d3` (the required rows and files are in `SUBJECT.md`).